In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
loss_history_dir = Path("../loss_history")
loss_history_paths = sorted(loss_history_dir.glob("loss_history_*.csv"))

if not loss_history_paths:
    raise FileNotFoundError(f"No loss history files found in {loss_history_dir.resolve()}")

loss_frames = []
for path in loss_history_paths:
    run_df = pd.read_csv(path)
    run_df["source_file"] = path.name
    if "run_id" not in run_df.columns:
        run_df["run_id"] = path.stem.replace("loss_history_", "")
    if "test_id" in run_df.columns:
        run_df["test_id"] = pd.to_numeric(run_df["test_id"], errors="coerce").astype("Int64")
    loss_frames.append(run_df)

loss_df = pd.concat(loss_frames, ignore_index=True)
sort_columns = [col for col in ["test_id", "run_id", "epoch"] if col in loss_df.columns]
loss_df = loss_df.sort_values(sort_columns).reset_index(drop=True)

available_runs = (
    loss_df[["test_id", "seed", "run_id", "source_file"]]
    .drop_duplicates()
    .sort_values([col for col in ["test_id", "run_id"] if col in loss_df.columns])
    .reset_index(drop=True)
)

test_case_runs = (
    available_runs.groupby("test_id", dropna=False)
    .agg(
        run_count=("run_id", "nunique"),
        seeds=("seed", lambda s: sorted({int(v) for v in s.dropna()})),
        latest_run_id=("run_id", "max"),
    )
    .reset_index()
    .sort_values("test_id", na_position="last")
)

print(f"Loaded {len(available_runs)} runs from {loss_history_dir.resolve()}")
test_case_runs


In [ ]:
available_runs


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

for run_id, run_df in loss_df.groupby("run_id", sort=True):
    run_label = f"run {run_id}"
    if "test_id" in run_df.columns and run_df["test_id"].notna().any():
        run_label = f"test {int(run_df['test_id'].dropna().iloc[0])} | {run_label}"
    if "seed" in run_df.columns and run_df["seed"].notna().any():
        run_label = f"{run_label} | seed {int(run_df['seed'].dropna().iloc[0])}"

    axes[0].plot(run_df["epoch"], run_df["total_loss"], label=f"total {run_label}", alpha=0.8)
    axes[0].plot(
        run_df["epoch"],
        run_df["wirelength_loss"],
        linestyle="--",
        label=f"wirelength {run_label}",
        alpha=0.7,
    )
    axes[0].plot(
        run_df["epoch"],
        run_df["overlap_loss"],
        linestyle=":",
        label=f"overlap {run_label}",
        alpha=0.7,
    )

    if "overlap_count" in run_df.columns:
        axes[1].plot(run_df["epoch"], run_df["overlap_count"], label=f"count {run_label}", alpha=0.8)
    if "total_overlap_area" in run_df.columns:
        axes[1].plot(
            run_df["epoch"],
            run_df["total_overlap_area"],
            linestyle="--",
            label=f"total area {run_label}",
            alpha=0.7,
        )
    if "max_overlap_area" in run_df.columns:
        axes[1].plot(
            run_df["epoch"],
            run_df["max_overlap_area"],
            linestyle=":",
            label=f"max area {run_label}",
            alpha=0.7,
        )

axes[0].set_ylabel("Loss")
axes[0].set_title("Loss History By Run")
axes[0].legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Overlap")
axes[1].set_title("Overlap Metrics By Run")
axes[1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
